# 02 — Baseline CNN Training
**Month 1, Week 3–4 Deliverable**

Goals:
1. Train the baseline CNN on within-hive train split
2. Get the first accuracy number
3. Plot training curves
4. Show confusion matrix
5. Document results as your Month 1 checkpoint


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import config
from data import get_dataloaders
from models import get_model

print('Setup complete.')


## Option A — Train from this notebook (small runs)

In [ ]:
# For full training, run from command line instead:
#   python train.py --model baseline_cnn --epochs 30 --experiment exp_01
#
# For a quick notebook run (fewer epochs, useful for debugging):

import torch
from train import train_one_epoch, evaluate, get_device, get_loss_fn
from torch.optim import Adam

device       = get_device()
train_loader, val_loader, test_loader = get_dataloaders()
model        = get_model('baseline_cnn').to(device)
criterion    = get_loss_fn(train_loader).to(device)
optimizer    = Adam(model.parameters(), lr=config.LEARNING_RATE)

# Quick 5-epoch run to verify everything works
history = {'train_acc': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, 6):
    train_m = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_m   = evaluate(model, val_loader, device, split_name='val')
    history['train_acc'].append(train_m['accuracy'])
    history['val_acc'].append(val_m['accuracy'])
    history['val_f1'].append(val_m['f1'])
    print(f'Epoch {epoch}: Train {train_m["accuracy"]:.4f} | Val {val_m["accuracy"]:.4f}')


## Load training log from CLI run

In [ ]:
import os
log_path = os.path.join(config.DATA_PROCESSED, 'exp_01_log.csv')

if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(log.tail())
else:
    print('No log found. Run: python train.py --model baseline_cnn --experiment exp_01')
    log = pd.DataFrame(history)  # use notebook history if available


## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(log['train_accuracy'], label='Train', color='#f59e0b')
ax1.plot(log['val_accuracy'],   label='Val',   color='#22c55e')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(log['train_loss'], label='Train Loss', color='#ef4444')
ax2.set_title('Training Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Baseline CNN — Training Curves', fontsize=13)
plt.tight_layout()
plt.show()


## Confusion Matrix

In [ ]:
import torch
from train import evaluate, get_device
from models import get_model
from data import get_dataloaders
import config

device      = get_device()
_, _, test_loader = get_dataloaders()

# Load best checkpoint
ckpt_path = os.path.join(config.CHECKPOINT_DIR, 'exp_01_best.pt')
if os.path.exists(ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=device)
    model = get_model('baseline_cnn').to(device)
    model.load_state_dict(ckpt['model_state'])
    test_m = evaluate(model, test_loader, device, split_name='test')
    
    cm = np.array(test_m['confusion_matrix'])
    labels = list(config.LABEL_MAP.keys())
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'Baseline CNN — Test Confusion Matrix\n(Acc: {test_m["accuracy"]*100:.1f}%)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
else:
    print('No checkpoint found. Train first.')


## Month 1 Checkpoint Summary

Fill in after running:
| Metric | Value |
|---|---|
| Within-hive train acc | ??? |
| Within-hive val acc   | ??? |
| Within-hive test acc  | ??? |
| Within-hive test F1   | ??? |

This number is your baseline. Month 2 (CRNN) should beat it on within-hive accuracy
and the cross-hive gap is your Month 2 headline result.
